# 05 — Cross-Feature Relationships

**Purpose:** measure how the feature blocks relate to each other and to future index returns before any regime model is fitted. Five questions from framework v2.1 are tested:

| Part | Question | Why it matters |
|---|---|---|
| A | How strongly do the underlyings co-move, and does BTC lead? | Pooling across assets; modelling thin assets conditionally on BTC/ETH (§14) |
| B | Which candidate inputs are redundant? | Overlapping inputs create spurious states; the emission must stay small |
| C | Does current activity predict volatility beyond trailing RV? | Notebook 04 proposed activity as a covariate of the structural scale $\sigma_{k,t}$ |
| D | Is implied volatility rich or cheap against the volatility that follows? | Sign of the variance risk premium decides whether the volatility book sells or buys options (§17.2) |
| E | How does flow move price, and does flow or OBI predict the next return? | Kyle's lambda for the exhaustion features (§5.4); directional content of the book |
| F | Does the next index return respond to displacement from an anchor, unconditionally? | First look at the quantity that defines the regimes (§7), with the mechanical-reversion trap made explicit |

Everything here is **unconditional or in-sample** unless stated otherwise. A pooled relationship mixes all regimes; finding none does not rule out regime-specific relationships, and finding one does not establish a regime. All timestamps are UTC.

## 1. Environment and read-only connection

In [ ]:
from pathlib import Path
import os
import math
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 220)
pd.set_option("display.width", 200)

load_dotenv()
data_root_env = os.getenv("DERIBIT_DATA_ROOT")
if not data_root_env:
    raise RuntimeError("DERIBIT_DATA_ROOT is not defined in the environment/.env file.")
DB_PATH = Path(data_root_env).expanduser().resolve() / "database" / "research.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"Research database not found: {DB_PATH}")

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")
print(f"Research DB: {DB_PATH} | time zone:", con.execute("SELECT current_setting('TimeZone')").fetchone()[0])

PERIODS_PER_YEAR = 365.0 * 24.0 * 12.0
MINUTES_PER_YEAR = 365.0 * 24.0 * 60.0

## 2. Load the state table

Requires `time_series_feature_state` schema 0.5.0 or later (perpetual integrity gate and clock variables). On an older table the notebook falls back to treating every perpetual state as valid and says so.

In [ ]:
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
WANTED = [
    "state_id", "state_time", "underlying", "session_id", "minutes_to_expiry", "is_opening_state",
    "return_5m", "realized_vol_1h_annualized", "realized_vol_30m_annualized",
    "log1p_volume_usd", "book_update_count", "trade_count", "signed_volume_usd", "trade_flow_imbalance",
    "volume_usd", "vwap", "futures_tradable_mid_price", "futures_index_price",
    "perpetual_mid_vs_index_bps", "futures_spread_bps", "book_twa_obi_20", "book_twa_spread_bps",
    "option_iv_atm_forward", "fit_atm_iv", "fit_skew_rel", "fit_curvature_rel", "rr_25d", "option_skew_025",
    "option_atm_quote_ok", "option_shape_quote_ok", "atm_iv_change_30m", "fit_atm_iv_change_30m",
    "perp_integrity_ok", "in_settlement_window", "utc_hour", "is_weekend", "expiry_type",
]
missing = [c for c in WANTED if c not in cols]
df = con.execute(f"SELECT {', '.join(c for c in WANTED if c in cols)} FROM time_series_feature_state ORDER BY underlying, state_time").df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
if missing:
    print("Missing columns:", missing)
if "perp_integrity_ok" not in df:
    print("perp_integrity_ok not found: rebuild with build_features.py (schema >= 0.5.0). Treating all perpetual states as valid.")
    df["perp_integrity_ok"] = True
df["perp_integrity_ok"] = df["perp_integrity_ok"].fillna(False).astype(bool)
for c in ["option_atm_quote_ok", "option_shape_quote_ok", "in_settlement_window"]:
    if c in df:
        df[c] = df[c].fillna(False).astype(bool)

ASSETS = sorted(df["underlying"].unique())
CORE = [a for a in ["BTC", "ETH"] if a in ASSETS]
TTE_EDGES = [0, 30, 60, 180, 360, 720, 1080, 1440]
TTE_LABELS = ["≤30m (settle)", "30m–1h", "1–3h", "3–6h", "6–12h", "12–18h", "18–24h"]
df["tte_bucket"] = pd.cut(df["minutes_to_expiry"], bins=TTE_EDGES, labels=TTE_LABELS, right=True)
df["log_updates"] = np.log1p(pd.to_numeric(df.get("book_update_count"), errors="coerce"))
print(f"Rows: {len(df):,} | assets: {ASSETS} | {df['state_time'].min()} → {df['state_time'].max()}")
print(f"Sessions: {df['session_id'].nunique()} | perpetual integrity ok: {100 * df['perp_integrity_ok'].mean():.1f}% of states")

In [ ]:
def grid_shift(d, column, k):
    """Value k bars away on the exact 5-minute grid, per asset (k>0: past, k<0: future)."""
    out = pd.Series(np.nan, index=d.index)
    for _, g in d.groupby("underlying"):
        g = g.sort_values("state_time")
        v = pd.to_numeric(g[column], errors="coerce").shift(k)
        ok = (g["state_time"] - g["state_time"].shift(k)).abs() == pd.Timedelta(minutes=5 * abs(k))
        out.loc[g.index] = v.where(ok)
    return out


def forward_sum(d, column, h):
    """Sum of the next h values (t+1..t+h) on the exact grid; NaN if any is missing."""
    total = pd.Series(0.0, index=d.index)
    for k in range(1, h + 1):
        total = total + grid_shift(d, column, -k)
    return total


def ols_hac(y, X, lags=12):
    """OLS with Newey–West standard errors. X without constant; returns DataFrame of coef, se, t."""
    d = pd.concat([y.rename("y"), X], axis=1).dropna()
    if len(d) < 50:
        return pd.DataFrame(), 0
    Y = d["y"].to_numpy()
    A = np.column_stack([np.ones(len(d)), d.drop(columns="y").to_numpy()])
    beta, *_ = np.linalg.lstsq(A, Y, rcond=None)
    u = Y - A @ beta
    XtX_inv = np.linalg.pinv(A.T @ A)
    S = (A * u[:, None]).T @ (A * u[:, None])
    for L in range(1, lags + 1):
        w = 1 - L / (lags + 1)
        G = (A[L:] * u[L:, None]).T @ (A[:-L] * u[:-L, None])
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    names = ["const"] + list(X.columns)
    return pd.DataFrame({"coef": beta, "se": se, "t": beta / np.where(se > 0, se, np.nan)}, index=names), len(d)

# Part A — Cross-Asset Co-Movement

## 3. Return and volatility correlation

Index returns are synchronous on the five-minute grid, so contemporaneous correlation measures common factors. Strong co-movement supports pooling parameters across assets; a weak link for an asset means its regimes cannot be borrowed from BTC/ETH.

In [ ]:
ret_wide = df.pivot_table(index="state_time", columns="underlying", values="return_5m")
ret_1h = ret_wide.rolling(12, min_periods=12).sum().iloc[::12]
rv_wide = df.pivot_table(index="state_time", columns="underlying", values="realized_vol_1h_annualized")

print("Spearman correlation of 5-minute index returns:")
display(ret_wide.corr(method="spearman").round(2))
print("Spearman correlation of non-overlapping 1-hour index returns:")
display(ret_1h.corr(method="spearman").round(2))
print("Spearman correlation of 1h realized volatility levels:")
display(rv_wide.corr(method="spearman").round(2))

## 4. Lead–lag

Correlation of each asset's return at $t+1$ with BTC's (and ETH's) return at $t$. A non-zero value means one index adjusts to another with a delay of at least one bar; the band printed below is the naive 95% band for zero.

In [ ]:
lead_rows = []
for leader in CORE:
    for asset in ASSETS:
        x = ret_wide[leader]
        for k in [1, 2, 3]:
            y = ret_wide[asset].shift(-k)
            ok = x.notna() & y.notna()
            lead_rows.append({"leader": leader, "follower": asset, "lag_bars": k, "n": int(ok.sum()),
                              "spearman": x[ok].corr(y[ok], method="spearman")})
lead_lag = pd.DataFrame(lead_rows)
display(lead_lag.pivot_table(index=["leader", "follower"], columns="lag_bars", values="spearman").round(3))
n0 = int(lead_lag["n"].median()) if len(lead_lag) else 1
print(f"Naive 95% band for zero at n≈{n0:,}: ±{1.96 / math.sqrt(max(n0, 1)):.3f}")

# Part B — Redundancy Among Candidate Inputs

## 5. Correlation structure

Candidate structural and exhaustion inputs, per asset, on states where the perpetual passes the integrity gate. The table lists pairs with median $|\rho| \ge 0.7$ across assets: within such a pair, one variable carries little information the other does not. Option features use quote-backed states only.

In [ ]:
CANDIDATES = [c for c in [
    "realized_vol_1h_annualized", "log1p_volume_usd", "log_updates", "trade_count", "trade_flow_imbalance",
    "perpetual_mid_vs_index_bps", "futures_spread_bps", "book_twa_spread_bps", "book_twa_obi_20",
    "option_iv_atm_forward", "fit_atm_iv", "fit_skew_rel", "fit_curvature_rel", "rr_25d", "option_skew_025",
] if c in df.columns]

work = df[df["perp_integrity_ok"]].copy()
for c in ["option_iv_atm_forward", "option_skew_025", "rr_25d"]:
    if c in work and "option_atm_quote_ok" in work:
        work.loc[~work["option_atm_quote_ok"], c] = np.nan

corrs = {}
for asset, g in work.groupby("underlying"):
    corrs[asset] = g[CANDIDATES].apply(pd.to_numeric, errors="coerce").corr(method="spearman", min_periods=200)
stack = pd.concat(corrs, names=["underlying", "feature"])
median_corr = stack.groupby(level="feature").median().reindex(index=CANDIDATES, columns=CANDIDATES)

pairs = []
for i, a in enumerate(CANDIDATES):
    for b in CANDIDATES[i + 1:]:
        vals = [corrs[x].loc[a, b] for x in corrs if not np.isnan(corrs[x].loc[a, b])]
        if vals:
            pairs.append({"feature_a": a, "feature_b": b, "median_rho": np.median(vals),
                          "min_rho": np.min(vals), "max_rho": np.max(vals), "assets": len(vals)})
redundancy = pd.DataFrame(pairs)
redundant = redundancy[redundancy["median_rho"].abs() >= 0.7].sort_values("median_rho", key=np.abs, ascending=False)
display(redundant.round(3))

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(median_corr.to_numpy(dtype=float), vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(CANDIDATES)), CANDIDATES, rotation=70, ha="right", fontsize=8)
ax.set_yticks(range(len(CANDIDATES)), CANDIDATES, fontsize=8)
ax.set_title("Median Spearman correlation across assets")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 6. Does activity forecast volatility beyond trailing RV?

Target: realized volatility over the **next** 30 minutes from index returns,

$$
\log \mathrm{RV}^{fwd}_{t} = \tfrac12 \log\!\left(\frac{\mathrm{PERIODS\_PER\_YEAR}}{6}\sum_{i=1}^{6} r_{t+i}^2\right)
$$

Nested models, per asset (HAR-style baseline, Corsi 2009):

- **M0:** $\log \mathrm{RV}_{1h,t}$, $\log \mathrm{RV}_{30m,t}$, $\log |r_t|$;
- **M1:** M0 + $\log(1+\text{volume}_t)$ + $\log(1+\text{book updates}_t)$;
- **M2:** M1 + $\log$ spread$_t$.

Fit on the first half of sessions and evaluated on the second half (out of sample). The gain in out-of-sample $R^2$ from M0 to M1 measures whether activity adds timely volatility information. Only states that pass the perpetual integrity gate enter M1/M2.

In [ ]:
d = df.copy()
d["r2"] = pd.to_numeric(d["return_5m"], errors="coerce") ** 2
d["fwd_rv30"] = np.sqrt(forward_sum(d, "r2", 6) / 6 * PERIODS_PER_YEAR)
d["y_log_fwd_rv"] = np.log(d["fwd_rv30"].where(d["fwd_rv30"] > 0))
d["x_log_rv1h"] = np.log(pd.to_numeric(d["realized_vol_1h_annualized"], errors="coerce").where(lambda s: s > 0))
d["x_log_rv30"] = np.log(pd.to_numeric(d.get("realized_vol_30m_annualized"), errors="coerce").where(lambda s: s > 0))
d["x_log_abs_r"] = np.log(pd.to_numeric(d["return_5m"], errors="coerce").abs().where(lambda s: s > 0))
d["x_log_volume"] = pd.to_numeric(d.get("log1p_volume_usd"), errors="coerce")
d["x_log_updates"] = d["log_updates"]
d["x_log_spread"] = np.log(pd.to_numeric(d.get("futures_spread_bps"), errors="coerce").where(lambda s: s > 0))

MODELS = {
    "M0_har": ["x_log_rv1h", "x_log_rv30", "x_log_abs_r"],
    "M1_activity": ["x_log_rv1h", "x_log_rv30", "x_log_abs_r", "x_log_volume", "x_log_updates"],
    "M2_spread": ["x_log_rv1h", "x_log_rv30", "x_log_abs_r", "x_log_volume", "x_log_updates", "x_log_spread"],
}

sessions = sorted(d["session_id"].dropna().unique())
rows, coefs = [], []
for asset, g in d[d["perp_integrity_ok"]].groupby("underlying"):
    sess = sorted(g["session_id"].unique())
    train_s = set(sess[: len(sess) // 2])
    cols_all = ["y_log_fwd_rv"] + MODELS["M2_spread"]
    g = g.dropna(subset=cols_all)
    tr, te = g[g["session_id"].isin(train_s)], g[~g["session_id"].isin(train_s)]
    if len(tr) < 200 or len(te) < 200:
        continue
    row = {"underlying": asset, "n_train": len(tr), "n_test": len(te)}
    for name, xs in MODELS.items():
        A = np.column_stack([np.ones(len(tr)), tr[xs].to_numpy()])
        b, *_ = np.linalg.lstsq(A, tr["y_log_fwd_rv"].to_numpy(), rcond=None)
        pred = np.column_stack([np.ones(len(te)), te[xs].to_numpy()]) @ b
        sse = ((te["y_log_fwd_rv"] - pred) ** 2).sum()
        sst = ((te["y_log_fwd_rv"] - tr["y_log_fwd_rv"].mean()) ** 2).sum()
        row[f"oos_r2_{name}"] = 1 - sse / sst
    rows.append(row)
    est, n = ols_hac(g["y_log_fwd_rv"], g[MODELS["M1_activity"]], lags=12)
    if len(est):
        coefs.append(est.assign(underlying=asset).reset_index(names="term"))
vol_forecast = pd.DataFrame(rows)
if len(vol_forecast):
    vol_forecast["gain_activity"] = vol_forecast["oos_r2_M1_activity"] - vol_forecast["oos_r2_M0_har"]
    vol_forecast["gain_spread"] = vol_forecast["oos_r2_M2_spread"] - vol_forecast["oos_r2_M1_activity"]
display(vol_forecast.round(4))
vol_coefs = pd.concat(coefs, ignore_index=True) if coefs else pd.DataFrame()
if len(vol_coefs):
    print("Full-sample M1 coefficients with Newey–West t-statistics (12 lags):")
    display(vol_coefs.pivot(index="term", columns="underlying", values="t").round(2))

# Part C — Implied Versus Realized

## 7. Variance risk premium to expiry

For each state, the quote-backed ATM implied volatility is compared with the volatility the index actually realized from $t$ to the start of the settlement window (07:30 UTC):

$$
\sigma^{R}_{t\to T} = \sqrt{\frac{\mathrm{PERIODS\_PER\_YEAR}}{n} \sum_{i=1}^{n} r_{t+i}^2}, \qquad
\log\frac{\sigma_{I,t}}{\sigma^R_{t\to T}}
$$

where $n$ is the number of remaining bars before 07:30. A positive median log ratio means options were priced above what followed (sellers were paid on average); this is the unconditional variance risk premium the volatility book of §17.2 would trade against.

Caveats: realized volatility over a few bars is very noisy; the last 30 minutes settle on an average (an Asian payoff with lower effective variance) and are excluded; quote-backed IV means the ATM node is bracketed by bid/ask-sourced contracts.

In [ ]:
v = df.copy()
v["r2"] = pd.to_numeric(v["return_5m"], errors="coerce") ** 2
v = v.sort_values(["underlying", "state_time"])
iv_col = "option_iv_atm_forward"
v["iv_dec"] = pd.to_numeric(v[iv_col], errors="coerce") / 100.0
v["remaining_bars"] = ((pd.to_numeric(v["minutes_to_expiry"], errors="coerce") - 30) // 5).clip(lower=0)

fwd_r2, fwd_n = [], []
for _, g in v.groupby(["underlying", "session_id"]):
    g = g.sort_values("state_time")
    r2 = g["r2"].to_numpy()
    rem = g["remaining_bars"].to_numpy()
    tot = np.full(len(g), np.nan)
    cnt = np.zeros(len(g))
    for i in range(len(g)):
        n_i = int(rem[i])
        seg = r2[i + 1: i + 1 + n_i]
        if n_i >= 6 and len(seg) == n_i and np.isfinite(seg).sum() >= 0.9 * n_i:
            tot[i] = np.nansum(seg) / np.isfinite(seg).sum()
            cnt[i] = np.isfinite(seg).sum()
    fwd_r2.append(pd.Series(tot, index=g.index))
    fwd_n.append(pd.Series(cnt, index=g.index))
v["fwd_var_per_bar"] = pd.concat(fwd_r2)
v["fwd_bars"] = pd.concat(fwd_n)
v["rv_to_expiry"] = np.sqrt(v["fwd_var_per_bar"] * PERIODS_PER_YEAR)
v["log_iv_over_rv"] = np.log(v["iv_dec"] / v["rv_to_expiry"].where(v["rv_to_expiry"] > 0))

vq = v[v["option_atm_quote_ok"] & v["log_iv_over_rv"].notna()] if "option_atm_quote_ok" in v else v[v["log_iv_over_rv"].notna()]
vrp = vq.groupby(["underlying", "tte_bucket"], observed=True).agg(
    states=("log_iv_over_rv", "size"),
    sessions=("session_id", "nunique"),
    median_iv=("iv_dec", "median"),
    median_rv_to_expiry=("rv_to_expiry", "median"),
    median_log_iv_over_rv=("log_iv_over_rv", "median"),
    share_iv_above_rv=("log_iv_over_rv", lambda s: (s > 0).mean()),
).round(3)
display(vrp)

# Session-level view: one observation per session at a fixed entry time, so states are not counted many times.
entry = vq[vq["minutes_to_expiry"].isin([1200, 720, 360])]
display(
    entry.pivot_table(index="underlying", columns="minutes_to_expiry", values="log_iv_over_rv",
                      aggfunc=["median", "count"]).round(3)
)

# Part D — Flow and Price

## 8. Price impact of signed flow (Kyle's lambda)

Same-bar regression of the index return on signed perpetual volume, per asset, on states that pass the integrity gate:

$$
r_t = \alpha + \lambda\, \frac{SV_t}{\overline{V}} + e_t
$$

with signed volume scaled by the asset's median absolute signed volume so that $\lambda$ is comparable across assets (bps of return per typical bar of net flow). A positive, stable $\lambda$ is a precondition for the exhaustion features of §5.4 (absorption is a falling $\lambda$). Stability is shown with a rolling 24-hour estimate.

In [ ]:
imp_rows, rolling = [], {}
for asset, g in df[df["perp_integrity_ok"]].groupby("underlying"):
    g = g.sort_values("state_time")
    sv = pd.to_numeric(g.get("signed_volume_usd"), errors="coerce")
    scale = sv.abs().median()
    if not scale or not np.isfinite(scale):
        continue
    x = (sv / scale).rename("signed_flow")
    y = 1e4 * pd.to_numeric(g["return_5m"], errors="coerce")
    est, n = ols_hac(y, x.to_frame(), lags=12)
    if len(est):
        imp_rows.append({"underlying": asset, "n": n, "lambda_bps": est.loc["signed_flow", "coef"],
                         "t_hac": est.loc["signed_flow", "t"],
                         "spearman": x.corr(y, method="spearman")})
    xy = pd.concat([x, y.rename("r")], axis=1).set_index(g["state_time"])
    cov = xy["signed_flow"].rolling(288, min_periods=144).cov(xy["r"])
    var = xy["signed_flow"].rolling(288, min_periods=144).var()
    rolling[asset] = cov / var
price_impact = pd.DataFrame(imp_rows)
display(price_impact.round(3))

fig, ax = plt.subplots(figsize=(13, 4.5))
for asset, s in rolling.items():
    ax.plot(s.index, s.values, linewidth=1, label=asset)
ax.axhline(0, linewidth=0.8)
ax.set_title("Rolling 24h price impact λ (bps per typical bar of net flow)")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
plt.show()

## 9. Does flow or the book predict the next return?

Correlation of next-bar index return $r_{t+1}$ with trade-flow imbalance, OBI and basis at $t$. Per framework §7, structural regimes are defined through the response to displacement, not through these; any unconditional predictability here is a candidate covariate, and its absence is expected for liquid assets at 5 minutes.

In [ ]:
p = df.copy()
p["next_return"] = grid_shift(p, "return_5m", -1)
pred_rows = []
for asset, g in p[p["perp_integrity_ok"]].groupby("underlying"):
    for col in ["trade_flow_imbalance", "book_twa_obi_20", "perpetual_mid_vs_index_bps", "return_5m"]:
        if col not in g:
            continue
        x = pd.to_numeric(g[col], errors="coerce")
        est, n = ols_hac(1e4 * g["next_return"], ((x - x.mean()) / x.std()).rename(col).to_frame(), lags=12)
        pred_rows.append({"underlying": asset, "feature": col, "n": n,
                          "spearman": x.corr(g["next_return"], method="spearman"),
                          "slope_bps_per_sd": est.loc[col, "coef"] if len(est) else np.nan,
                          "t_hac": est.loc[col, "t"] if len(est) else np.nan})
predictability = pd.DataFrame(pred_rows)
display(predictability.pivot(index="feature", columns="underlying", values="t_hac").round(2))
display(predictability.pivot(index="feature", columns="underlying", values="slope_bps_per_sd").round(3))

# Part E — Displacement Response (Preview)

## 10. Unconditional response of price to displacement

Displacement from an anchor $A_t$ (framework §1.3):

$$
D_t = \frac{\log I_t - \log A_t}{\hat\sigma_t \sqrt{h}}, \qquad \hat\sigma_t = \text{trailing per-bar volatility (1h RV, de-annualized)}
$$

Anchors:

- **index EWMA** with half-lives of 6, 12 and 24 bars (30 minutes, 1 hour, 2 hours): $\log A_t = \lambda \log A_{t-1} + (1-\lambda)\log I_t$, restarted at each session;
- **session VWAP** of the perpetual since 08:00 UTC, compared with the perpetual mid (same instrument, so the basis cancels); only states passing the integrity gate.

Two regressions per anchor and asset, with Newey–West standard errors:

$$
r_{t+1} = a + \beta D_t + e_{t+1} \qquad\text{(price response — informative)}
$$

$$
\Delta D_{t+1} = c + \gamma D_t + u_{t+1} \qquad\text{(displacement change — mechanical)}
$$

Under a random walk $\beta = 0$, but $\gamma < 0$ anyway because the anchor moves toward price. Comparing the two shows why states must be labelled by $\beta$, never by $\gamma$. The unconditional $\beta$ mixes all regimes: a value near zero is exactly what offsetting mean-reverting and momentum regimes would also produce.

In [ ]:
def ewma_anchor(g, half_life):
    lam = 0.5 ** (1.0 / half_life)
    logi = np.log(pd.to_numeric(g["futures_index_price"], errors="coerce"))
    out = np.full(len(g), np.nan)
    prev = np.nan
    prev_sess = None
    prev_time = None
    for i, (li, sess, ts) in enumerate(zip(logi.to_numpy(), g["session_id"].to_numpy(), g["state_time"].to_numpy())):
        contiguous = prev_time is not None and (ts - prev_time) == np.timedelta64(5, "m")
        if not np.isfinite(li) or sess != prev_sess or not contiguous or not np.isfinite(prev):
            prev = li
        else:
            prev = lam * prev + (1 - lam) * li
        out[i] = prev
        prev_sess, prev_time = sess, ts
    return out


disp_rows = []
x = df.sort_values(["underlying", "state_time"]).copy()
x["sigma_bar"] = pd.to_numeric(x["realized_vol_1h_annualized"], errors="coerce") / math.sqrt(PERIODS_PER_YEAR)
x["next_return"] = grid_shift(x, "return_5m", -1)

for asset, g in x.groupby("underlying"):
    g = g.copy()
    logi = np.log(pd.to_numeric(g["futures_index_price"], errors="coerce"))
    anchors = {}
    for hl in [6, 12, 24]:
        a = ewma_anchor(g, hl)
        anchors[f"ewma_hl{hl}"] = ((logi - a) / (g["sigma_bar"] * math.sqrt(hl)), np.ones(len(g), dtype=bool))
    if {"vwap", "volume_usd", "futures_tradable_mid_price"}.issubset(g.columns):
        vw = pd.to_numeric(g["vwap"], errors="coerce")
        vol = pd.to_numeric(g["volume_usd"], errors="coerce").fillna(0)
        num = (vw.fillna(0) * vol).groupby(g["session_id"]).cumsum()
        den = vol.groupby(g["session_id"]).cumsum()
        sess_vwap = (num / den.where(den > 0))
        bars_in = g.groupby("session_id").cumcount() + 1
        logm = np.log(pd.to_numeric(g["futures_tradable_mid_price"], errors="coerce"))
        anchors["session_vwap"] = ((logm - np.log(sess_vwap)) / (g["sigma_bar"] * np.sqrt(bars_in)),
                                   g["perp_integrity_ok"].to_numpy() & (bars_in >= 12).to_numpy())
    for name, (D, valid) in anchors.items():
        Dv = pd.Series(np.where(valid, D, np.nan), index=g.index).replace([np.inf, -np.inf], np.nan)
        g["_D"] = Dv
        dD = grid_shift(g, "_D", -1) - Dv
        same_sess = g["session_id"] == g["session_id"].shift(-1)
        est_r, n_r = ols_hac(1e4 * g["next_return"].where(same_sess), Dv.rename("D").to_frame(), lags=12)
        est_d, n_d = ols_hac(dD.where(same_sess), Dv.rename("D").to_frame(), lags=12)
        if len(est_r) and len(est_d):
            disp_rows.append({"underlying": asset, "anchor": name, "n": n_r,
                              "beta_bps_per_unit_D": est_r.loc["D", "coef"], "t_beta": est_r.loc["D", "t"],
                              "gamma_dD": est_d.loc["D", "coef"], "t_gamma": est_d.loc["D", "t"],
                              "sd_D": Dv.std()})
displacement = pd.DataFrame(disp_rows)
display(displacement.pivot(index="anchor", columns="underlying", values="t_beta").round(2))
display(displacement.pivot(index="anchor", columns="underlying", values="gamma_dD").round(3))
display(displacement.round(3))

## 11. Response by displacement size

A regression slope assumes a straight line. Mean reversion that appears only at large displacements (or momentum only at small ones) would be averaged away. The table bins $D_t$ (index EWMA, 1-hour half-life) into quantile bins and reports the mean next-bar return in basis points, per asset.

In [ ]:
bins_rows = []
for asset, g in x.groupby("underlying"):
    g = g.copy()
    a = ewma_anchor(g, 12)
    D = (np.log(pd.to_numeric(g["futures_index_price"], errors="coerce")) - a) / (g["sigma_bar"] * math.sqrt(12))
    nr = 1e4 * g["next_return"].where(g["session_id"] == g["session_id"].shift(-1))
    t = pd.DataFrame({"D": D, "nr": nr}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(t) < 500:
        continue
    t["bin"] = pd.qcut(t["D"], 10, labels=False, duplicates="drop")
    s = t.groupby("bin").agg(mean_D=("D", "mean"), mean_next_bps=("nr", "mean"), se=("nr", lambda v: v.std() / math.sqrt(len(v))), n=("nr", "size"))
    s["underlying"] = asset
    bins_rows.append(s.reset_index())
disp_bins = pd.concat(bins_rows, ignore_index=True) if bins_rows else pd.DataFrame()
if len(disp_bins):
    display(disp_bins.pivot(index="bin", columns="underlying", values="mean_next_bps").round(2))
    fig, ax = plt.subplots(figsize=(10, 5))
    for asset, s in disp_bins.groupby("underlying"):
        ax.errorbar(s["mean_D"], s["mean_next_bps"], yerr=1.96 * s["se"], marker="o", ms=3, capsize=2, label=asset)
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("displacement D (index EWMA, 1h half-life)")
    ax.set_ylabel("mean next 5m index return (bps)")
    ax.set_title("Next-bar return by displacement decile (±1.96 SE, naive)")
    ax.legend(fontsize=8, ncol=4)
    plt.tight_layout()
    plt.show()

# Part F — Research Handoff

## 12. Interpretation checklist

1. **Co-movement:** how correlated are the alts with BTC/ETH at 5 minutes and 1 hour? Does BTC lead any asset by one bar or more?
2. **Redundancy:** which pairs exceed $|\rho| \ge 0.7$? Which member of each pair should represent it in the emission?
3. **Activity and volatility:** does adding volume and book updates raise out-of-sample $R^2$ for the next 30-minute RV? For which assets?
4. **Variance risk premium:** is quote-backed ATM IV above subsequently realized volatility, by how much, and does it depend on time to expiry? How many sessions support it?
5. **Price impact:** is $\lambda$ positive and stable enough to use its changes as an exhaustion signal?
6. **Flow predictability:** does any of flow imbalance, OBI, basis or the last return predict the next return (HAC $|t| > 2$)?
7. **Displacement:** is the unconditional $\beta$ distinguishable from zero for any anchor? Is $\gamma$ negative everywhere, confirming the mechanical reversion?
8. **Non-linearity:** does the decile plot show reversion or continuation only in the tails?
9. **History:** with 17 sessions, which of these conclusions are fragile?

These are unconditional, in-sample relationships. They inform the feature contract (Notebook 07) and the baselines and null simulation (framework Step 4); they do not establish regimes.

In [ ]:
cross_feature_report = {
    "lead_lag": lead_lag,
    "redundancy": redundancy,
    "vol_forecast": vol_forecast,
    "vol_coefs": vol_coefs,
    "vrp": vrp.reset_index(),
    "price_impact": price_impact,
    "predictability": predictability,
    "displacement": displacement,
    "displacement_bins": disp_bins,
}
for k, val in cross_feature_report.items():
    print(f"  {k:<18} {len(val):>7,} rows")

## 13. Close the read-only connection

In [ ]:
con.close()
print("DuckDB connection closed.")